#### Workflow Objectives: 
The primary focus of this phase is to ensure high-quality raw data and accurate mapping to the reference genome. The workflow utilizes three core stages:

* **Initial QC**: Perform FastQC (v0.12.1) to evaluate per-base sequence quality, GC content, and adapter contamination. Monitor for Tn5-induced sequence bias and PCR duplication.
* **Read Trimming**: Execute Trim Galore (v0.6.10) in paired-end mode ($Q > 20$, length $> 70$ bp) to remove adapters and low-quality bases.
* **Alignment**: Map processed reads to the GRCh38.p13 reference genome using Bowtie2 (v2.4.5) with --very-sensitive and a fragment limit of $2000$ bp (-X 2000).

#### Data Processing & Implementation
All tasks are submitted to the HPC cluster using the **SLURM scheduler** (default: 4 GB memory, 1 CPU, 6-hour runtime). Each step is executed as a separate job script to ensure efficient resource allocation and reproducibility.  

#### Step 1: Pre-alignment quality control 
Initial FastQC and MultiQC run on all fastq files


In [ ]:
#!/bin/sh

# Runs FastQC on all FASTQ files before trimming

# Assumes:
#  - FASTQ files are in ./fastq_files
#  - FastQC is installed at ~/tools/FastQC/fastqc (edit if different)
#  - multiqc is available in the conda environment

# Create output directory
mkdir -p ./fastqc_results

# Run FastQC on all gzipped FASTQ files
# Output reports (.html + .zip) will be written to fastqc_results
~/tools/FastQC/fastqc ./fastq_files/*.fastq.gz -o ./fastqc_results

# Run MultiQC
conda activate python3.7
multiqc ./fastqc_results -o ./fastqc_results
conda deactivate

**1.1 Read quality**

Confirm that base quality scores remain consistently high (≥Q30), particularly at the start of reads. A gradual drop in quality toward the read ends is typical for Illumina sequencing and generally expected.

<img src="https://www.dropbox.com/scl/fi/exnzbadlfm9m5auuw0b16/fastqc_per_base_sequence_quality_plot-8.png?rlkey=rlqcck7vyijm5wtjhze0qm01h&st=xag2whcl&raw=1" width="500" alt="read_quality">

* High accuracy: All samples maintain mean Phred >Q30 for most of the read length (0–110 bp).
* Typical 3' decay: Quality gradually drops at 120–150 bp but stays above Q24 (99.6% accuracy).


**1.2 Per base sequence content**   

Show the per-base composition across read positions. The proportions of A, T, C, and G should follow an approximately linear, parallel trend across the read. Note that slight deviations at the start can reflect Tn5 insertion bias, since Nextera library prep uses a modified Tn5 transposase to fragment chromatin and add sequencing adapters.

<img src="https://www.dropbox.com/scl/fi/b381n7jhhtwvdydpe576g/per_base_sequence_content.PNG?rlkey=4t4t3x2r2pj4zvw89fcv74da2&st=9efbv1jk&raw=1" width="1500" alt="read_quality">

* Observation: First ~9–12 bp show non-random nucleotide patterns (vertical banding).
* Interpretation: Expected due to Tn5 sequence preference; confirms successful transposition.


**1.3 Per base GC content** 

Chromatin-accessible regions can show non-uniform GC content, so some deviation from a normal distribution is expected in ATAC-seq. However, pronounced GC bias may signal technical artifacts such as PCR amplification bias. Similarly, sharp spikes within an otherwise smooth GC distribution often point to specific contaminants, for example adapter dimers.

<img src="https://www.dropbox.com/scl/fi/h5ime7xz53ychkbyy5fm3/fastqc_per_sequence_gc_content_plot-11.png?rlkey=5ov26ban7isgdqufwi863n6qu&st=13l6ir62&raw=1" width="500" alt="read_quality">

* Bimodal Pattern: Most samples peak at ~50–52% GC, while a subset peaks ~60%.
* Implication: Indicates either true biological differences or potential contamination/library issues if all samples are expected to be uniform.

**1.4 Adapter content**  

Because ATAC-seq libraries contain very short DNA fragments, sequencing reads frequently extend into adapter sequence, particularly for the shortest inserts. As a result, detection of Nextera transposase/adapter sequence in read tails is common and typically requires adapter trimming before downstream analysis.

<img src="https://www.dropbox.com/scl/fi/w7k5s4iwl634sj0eh6gae/fastqc_adapter_content_plot-9.png?rlkey=qces0ri51t7epxroyci4kg00r&st=gr33wd20&raw=1" width="500" alt="read_quality">

* Severe Contamination: Adapter content rises sharply from 35–40 bp, reaching 40–50% by 150 bp.
* Cause: Short insert sizes caused reads to extend into adapters, producing the classic "hockey stick" pattern.

**1.5 Sequence duplication level**  

Some duplication is expected in ATAC-seq data due to both technical and biological factors:
* PCR bias: Library fragments can be over-amplified during PCR, leading to technical duplicates.
* Biological enrichment: Highly accessible chromatin regions are sequenced repeatedly, producing genuine duplicates.  

However, duplication rates above ~50% may indicate suboptimal library complexity or over-amplification during PCR, where a limited set of fragments becomes disproportionately represented.

<img src="https://www.dropbox.com/scl/fi/sabw6lqfgu627jug4r0ju/fastqc_sequence_duplication_levels_plot-9.png?rlkey=szfna8hmt2pu7di1qldlfetze&st=326ipdx8&raw=1" width="500" alt="read_quality">

* High complexity with ~65–70% unique reads; duplicated reads drop sharply—expected for quality libraries.
* Lower complexity (~35–45% unique), with elevated high-duplication reads (>10), indicating potential batch effects or mitochondrial DNA contamination.


### Step 2. Read Trimming 
Trimming was automated via SLURM array jobs. This stage effectively removed the Nextera Transposase sequences identified in the initial QC.

2a. Run TrimGalore

In [ ]:
#!/bin/sh
# Run trimgalore for automatic adapter decontamination (paired-end)
# Assumes:
#  - cutadaptenv is available in the conda environment

#SBATCH --cpus-per-task=4
#SBATCH --mem=8G
#SBATCH -o trimgalore_results/slurm_%A_%a.out
#SBATCH -e trimgalore_results/slurm_%A_%a.err

# Create output directory
mkdir trimgalore_results

# Activate conda environment containing TrimGalore/cutadapt
conda activate cutadaptenv

# Build list of sample prefixes from FASTQ files
# Assumes files like: sample_R1_001.fastq.gz and sample_R2_001.fastq.gz

# Define an array of file names to be processed
readarray -t files < <(ls fastq_files/*.fastq.gz | sed 's/.*fastq_files//' | sed 's/_R.*//' | sort -u)

# Run TrimGalore (paired-end)
file="${files[$SLURM_ARRAY_TASK_ID - 1]}"

# Perform bowtie2 alignment for the specific file
srun ~/tools/TrimGalore-0.6.10/trim_galore -q 20 --phred33 --length 70 -o ./trimgalore_results --paired fastq_files${file}_R1_001.fastq.gz fastq_files${file}_R2_001.fastq.gz 

# Deactivate conda environment
conda deactivate

2b. Run FastQC and MultiQC on trimmomatic generated fastq files

In [ ]:
#!/bin/sh

# Runs FastQC on all FASTQ files after trimming

# Create output directory
mkdir -p ./fastqc_results

# Run FastQC 
# Output reports (.html + .zip) will be written to trimgalore_results
~/tools/FastQC/fastqc ./trimgalore_results/*.fq.gz -o ./trimgalore_results

# Run MultiQC
conda activate python3.7
multiqc ./fastqc_results -o ./fastqc_results
conda deactivate

#### Step 3: Alignment (Bowtie2)
Reads were mapped to the GRCh38 assembly. Genome indexes were generated for both ENSEMBL and UCSC annotations to ensure flexibility in downstream annotation.

3a. Generate Genome indexes. 
This process needs to be done once per genome.

In [ ]:
#!/bin/sh
# Generate Bowtie2 genome indexes for human genome (hg38)
# This script downloads reference FASTA files and builds Bowtie2 indexes
# Both ENSEMBL and UCSC versions are included

# Load Bowtie2 module (required if using a module-based system)
module load Bowtie2/

# Create output directory and move into it
mkdir bowtie2
cd bowtie2 

# 1. ENSEMBL annotation
# Downloads the primary assembly FASTA from ENSEMBL and builds index
wget https://ftp.ensembl.org/pub/release-110/fasta/homo_sapiens/dna/Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz

# Build Bowtie2 index for ENSEMBL reference
# Output files will be prefixed with "Homo_sapiens"
bowtie2-build Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz Homo_sapiens

# 2. UCSC annotation
# Downloads hg38 FASTA from UCSC and builds index
wget https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz

# Build Bowtie2 index for UCSC reference
# Output files will be prefixed with "hg38"
bowtie2-build hg38.fa.gz hg38

3b. Read alignment with bowtie2 (hg38 assembly)

In [ ]:
#!/bin/bash
# Perform Bowtie2 alignment for trimmed paired-end FASTQ files

#SBATCH --cpus-per-task=8
#SBATCH --mem=32G
#SBATCH -t 12:00:00
#SBATCH -o bowtie2_results/slurm_%A_%a.out
#SBATCH -e bowtie2_results/slurm_%A_%a.err

# Load Bowtie2 module (required if using a module-based system)
module load Bowtie2

# Create the output directory
mkdir -p bowtie2_results

# Define an array of file names to be processed
readarray -t files < <(ls fastq_files/*.fastq.gz | sed 's/.*fastq_files//' | sed 's/_R.*//' | sort -u)

# Get the file name for this array task
file="${files[$SLURM_ARRAY_TASK_ID - 1]}"

# Perform bowtie2 alignment for the specific file
srun bowtie2 --very-sensitive -X 2000 -x ./bowtie2/hg38 \
    -1 trimgalore_results${file}_R1_001_val_1.fq.gz \
    -2 trimgalore_results${file}_R2_001_val_2.fq.gz \
    -S bowtie2_results/${file}.sam

# Run MultiQC
conda activate python3.7
multiqc ./bowtie2_results/. -o ./bowtie2_results/
conda deactivate 

<img src="https://www.dropbox.com/scl/fi/ipod05w0m33wd5r76gwm4/bowtie2_stats.PNG?rlkey=6b5b9dkrt1sn5a6mwgkdyg52r&st=bmlwf6ce&raw=1" width="1000" alt="read_quality">

* High performance: Most samples show >80% unique alignment (dark blue).
* Consistency: Multimapping rates are stable at 10–15% (orange).
* Quality flag: One outlier shows low alignment and high unmapped reads (dark red), but 20.2 M reads (72.0%) are uniquely and concordantly aligned, generally sufficient for standard ATAC-seq analysis.

#### Summary of Observations
* **Data Integrity**: The trimming process successfully resolved the adapter contamination identified by MultiQC.
* **Computational Efficiency**: Utilizing array tasks reduced total wall-clock time for multi-sample batch.
* **Next Steps**: Remove reads mapping to chrM/MT to address duplication and GC bias. Re-evaluate library complexity (PCR bottlenecking) on filtered BAMs to confirm sample suitability for peak calling.

#### Resources
* https://github.com/nf-core/atacseq
* https://doi.org/10.1186/s13059-020-1929-3
* https://bioinformaticamente.com/2024/12/05/comprehensive-guide-to-atac-seq-data-quality-control/